# 00 · Environment Setup

Before running any other notebook in this course, set up the project's Python environment with
[uv](https://docs.astral.sh/uv/) — a fast, all-in-one package/project manager that replaces
`pip` + `venv` + `pip-tools`. `pyproject.toml` declares the dependencies; `uv sync` resolves them,
writes a lockfile (`uv.lock`), and installs everything into an isolated `.venv/`.

> **You can run *this* notebook with any Python 3 kernel** (even a bare system one) — it only
> shells out to the `uv` CLI. Once it finishes, switch the kernel for notebooks 01+ to the one
> this notebook registers.

> **Planning to run the full 95-patient cohort? Use a server, not Colab.** Every notebook here
> opens in Colab, which is fine for reading along or a small-subset pass. But the full CHIMERA
> Task 1 cohort is several GB of gigapixel whole-slide images, patch and feature extraction are
> I/O- and GPU-heavy, and training all three fusion strategies across every CV fold runs long —
> more than Colab's disk limits, session timeouts, and shared GPUs comfortably handle. For a full
> run, clone the repo onto a machine or cluster with real storage and a GPU:
> ```bash
> git clone https://github.com/yws0322/minicourse-multimodal.git
> cd minicourse-multimodal
> ```
> then continue with this notebook there.

## Step 1 · Install uv

If `uv` isn't installed yet:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh    # macOS / Linux
# or: pipx install uv
# or: brew install uv
```

Check it from here:

In [ ]:
import shutil
import subprocess

if shutil.which("uv") is None:
    raise RuntimeError(
        "uv not found on PATH. Install it first -- see the cell above -- then restart this kernel."
    )

print(subprocess.run(["uv", "--version"], capture_output=True, text=True).stdout.strip())

## Step 2 · Resolve and install dependencies

`uv sync` reads `pyproject.toml`, resolves a consistent set of package versions, writes/updates
`uv.lock`, and installs everything into `PROJECT_ROOT/.venv`. `--project` points uv at the repo
root regardless of where this notebook's working directory happens to be (`notebooks/`).

This installs both the lightweight packages needed starting now (numpy, pandas, scikit-learn,
openslide-python, ...) and the heavier ones only needed from notebook 02 onward (torch, timm,
SimpleITK, ...) — so this one sync covers the whole course.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> repo root
assert (PROJECT_ROOT / "pyproject.toml").exists(), f"Expected pyproject.toml under {PROJECT_ROOT}"

result = subprocess.run(
    ["uv", "sync", "--project", str(PROJECT_ROOT)],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

**GPU note:** the default `uv sync` installs whatever PyTorch wheel uv resolves for your platform
(often CPU-only, or whatever CUDA version is the current default). If you need a specific CUDA
build for the GPU you're running on, install it explicitly afterwards, e.g.:

```bash
uv pip install torch --index-url https://download.pytorch.org/whl/cu121 --project <repo root>
```

**OpenSlide:** WSI patching/feature extraction needs the OpenSlide C library, which
`openslide-python` only binds to. Rather than requiring a separate system install (`apt`, `brew`,
cluster modules — often unavailable/inconvenient on shared clusters), `pyproject.toml` also
includes `openslide-bin`, which bundles a prebuilt copy — `uv sync` alone is enough.

## Step 3 · Register the environment as a Jupyter kernel

So that notebooks 01+ can actually *use* `.venv` from Jupyter, register it as a named kernel:

In [ ]:
KERNEL_NAME = "minicourse-multimodal"

result = subprocess.run(
    [
        "uv", "run", "--project", str(PROJECT_ROOT),
        "python", "-m", "ipykernel", "install", "--user",
        "--name", KERNEL_NAME, "--display-name", f"{KERNEL_NAME} (uv)",
    ],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

Now switch **this notebook's kernel** (and every notebook from here on) to
**"minicourse-multimodal (uv)"** — Kernel menu → Change Kernel in Jupyter, or the kernel picker in
VS Code / JupyterLab — then run the verification cell below.

## Step 4 · Verify

Run this *after* switching to the new kernel. `sys.executable` should point inside
`PROJECT_ROOT/.venv`, and every import should succeed.

In [ ]:
import sys

print("Python:", sys.executable)

import numpy, pandas, sklearn, matplotlib, PIL
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("scikit-learn", sklearn.__version__)
print("matplotlib", matplotlib.__version__)
print("Pillow", PIL.__version__)

import openslide
print("openslide-python", openslide.__version__)

In [ ]:
# Needed starting in notebook 02 -- fine to run now too.
import torch, torchvision, timm
import SimpleITK as sitk
import pytorch_lightning as pl

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("torchvision", torchvision.__version__)
print("timm", timm.__version__)
print("SimpleITK", sitk.Version_VersionString())
print("pytorch-lightning", pl.__version__)

## Step 5 · HuggingFace token (needed for notebook 02)

Notebook 02 downloads gated foundation-model weights (H-optimus-0) from HuggingFace, which needs
an access token. This project keeps secrets in a gitignored `.env` file rather than requiring an
interactive `huggingface-cli login` every time:

1. Get a token at https://huggingface.co/settings/tokens (read access is enough), and make sure
   your account has been granted access to https://huggingface.co/bioptimus/H-optimus-0.
2. Copy `.env.example` to `.env` (if you haven't already) and paste your token in:
   ```bash
   cp .env.example .env   # then edit .env, filling in HF_TOKEN=...
   ```
3. `.env` is gitignored — it never gets committed. `.env.example` (committed) just documents
   which keys are expected.

Notebook 02 loads it with `python-dotenv` and logs in programmatically — see it verified below.

In [ ]:
import os
from dotenv import load_dotenv

env_path = PROJECT_ROOT / ".env"
if not env_path.exists():
    print(f"No .env found at {env_path} -- copy .env.example to .env and fill in HF_TOKEN.")
else:
    load_dotenv(env_path)
    token_set = bool(os.environ.get("HF_TOKEN"))
    print(f".env loaded from {env_path}")
    print("HF_TOKEN set:" , token_set, "(fine to leave blank for now -- only needed for notebook 02)")

## Summary

- `pyproject.toml` — declares every dependency for the whole course
- `uv.lock` — the resolved, locked versions (commit this; don't hand-edit it)
- `.venv/` — the actual installed environment (gitignored)
- A registered Jupyter kernel, **"minicourse-multimodal (uv)"**, for notebooks 01+

**Adding a dependency later:** run `uv add <package> --project <repo root>` — it updates
`pyproject.toml` and `uv.lock` and installs it, instead of hand-editing either file.

**Next up — `01_data_preparation.ipynb`:** make sure its kernel is also set to
"minicourse-multimodal (uv)", then start there.